In [ ]:
import tkinter as tk
from tkinter import ttk
from datetime import datetime, timedelta
import pytz
import os
import sys
import subprocess
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# --- Configuration (Move to top for easy access) ---

# Replace with your actual credentials and sheet URL.  For testing, you might
# use a small, sample spreadsheet.
# BEST PRACTICE:  Store these *outside* your code (environment variables, config file).
# Never hardcode credentials directly in the script, especially if you share it.
scope = [
    "https://spreadsheets.google.com/feeds",
    'https://www.googleapis.com/auth/spreadsheets',
    "https://www.googleapis.com/auth/drive.file",
    "https://www.googleapis.com/auth/drive"
]

json_path = resource_path("service_account.json")
sheet_url = "YOUR_SHEET_URL"  # Replace with your sheet URL
fertilizer_type = ["NPK 13", "NPK 15", "NPK 12", "Dolomite", "Urea", "MOP", "HGFB", "CuSO4", "Zincop Chelated", "Kieserite", "RP", "Kaptan", "TSP"]

# --- Utility Functions ---

def format_datetime(dt):
    """Formats a datetime object to 'dd/mm/yyyy'."""
    return dt.strftime('%d/%m/%Y') if isinstance(dt, datetime) else ''

def format_datetimehour(dt):
    """Formats a datetime object to 'dd/mm/yyyy HH:MM:SS'."""
    return dt.strftime('%d/%m/%Y %H:%M:%S') if isinstance(dt, datetime) else ''

def resource_path(relative_path):
    """Gets the absolute path to a resource (for PyInstaller compatibility)."""
    try:
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)


# --- Spreadsheet Interaction Functions ---

def load_data(sheet_url, json_path):
    """Loads data from the Google Sheet into a Pandas DataFrame."""
    try:
        creds = ServiceAccountCredentials.from_json_keyfile_name(json_path, scope)
        client = gspread.authorize(creds)
        sheet_data = client.open_by_url(sheet_url).worksheet("DB")
        data = sheet_data.get_all_records()
        df = pd.DataFrame(data)

        # Data type conversions and cleaning
        df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
        df['Daily Rainfall (mm)'] = pd.to_numeric(df['Daily Rainfall (mm)'], errors='coerce')
        df.dropna(subset=['Date'], inplace=True)  # Drop rows with invalid dates

        return df

    except gspread.exceptions.SpreadsheetNotFound:
        print(f"Error: Spreadsheet not found at URL: {sheet_url}")
        return pd.DataFrame()  # Return an empty DataFrame on error
    except gspread.exceptions.APIError as e:
        print(f"Error: API Error accessing Google Sheets: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"An unexpected error occurred loading data: {e}")
        return pd.DataFrame()

def update_rainfall_data(df, selected_estate, new_rainfall, date):
    """Updates the rainfall data in the DataFrame and uploads it to the sheet."""
    global sheet_url, json_path

    # Convert date to datetime object with correct format
    try:
        date = pd.to_datetime(date, format="%d/%m/%Y")
    except ValueError:
        tk.messagebox.showerror("Error", "Invalid date format.  Please use DD/MM/YYYY.")
        return False # Indicate failure

    # Input validation
    if not isinstance(new_rainfall, (int, float)):
        try:
            new_rainfall = float(new_rainfall)
        except ValueError:
             tk.messagebox.showerror("Error", "Invalid rainfall value. Please enter a number.")
             return False

    # Find the last record for the selected estate
    estate_df = df[(df['Estate'] == selected_estate)]
    if estate_df.empty: # Handle cases where estate isn't found
        tk.messagebox.showinfo("Info", f"No data found for {selected_estate}. Adding a new entry.")
        new_row = {'Estate':selected_estate, 'Date': date, 'Daily Rainfall (mm)':new_rainfall}
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    else:
         # Find index of last entry for the estate.
        last_index = estate_df.index[-1]
        # Update DataFrame
        df.loc[last_index, 'Daily Rainfall (mm)'] = new_rainfall
        df.loc[last_index, 'Date'] = date # Update the date as well


    # Upload the updated DataFrame to the Google Sheet
    try:
        creds = ServiceAccountCredentials.from_json_keyfile_name(json_path, scope)
        client = gspread.authorize(creds)
        sheet_output = client.open_by_url(sheet_url).worksheet("Output")  # Use "Output" sheet
        sheet_output.clear() # Clear the sheet first!  Important!
        sheet_output.update([df.columns.values.tolist()] + df.values.tolist())
        tk.messagebox.showinfo("Success", "Rainfall data updated successfully!")
        return True

    except Exception as e:
        tk.messagebox.showerror("Error", f"Failed to update Google Sheet: {e}")
        return False

def add_rainfall_data(df, selected_estate, new_rainfall, date):
    """Adds new rainfall data to the DataFrame and uploads it."""
    global sheet_url, json_path

    # Convert date to datetime object with correct format
    try:
        date = pd.to_datetime(date, format="%d/%m/%Y")
    except ValueError:
        tk.messagebox.showerror("Error", "Invalid date format.  Please use DD/MM/YYYY.")
        return False # Indicate failure


    # Input Validation (same as in update_rainfall_data)
    if not isinstance(new_rainfall, (int, float)):
        try:
            new_rainfall = float(new_rainfall)
        except ValueError:
            tk.messagebox.showerror("Error", "Invalid rainfall value. Please enter a number.")
            return False


    # Create a new row as a dictionary
    new_row = {'Estate': selected_estate, 'Date': date, 'Daily Rainfall (mm)': new_rainfall}

    # Append to DataFrame.  Use pandas.concat for efficiency.
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


    # Upload to Google Sheets (same as update_rainfall_data)
    try:
        creds = ServiceAccountCredentials.from_json_keyfile_name(json_path, scope)
        client = gspread.authorize(creds)
        sheet_output = client.open_by_url(sheet_url).worksheet("Output")
        sheet_output.clear()
        sheet_output.update([df.columns.values.tolist()] + df.values.tolist())
        tk.messagebox.showinfo("Success", "Rainfall data added successfully!")
        return True  # Indicate success

    except Exception as e:
        tk.messagebox.showerror("Error", f"Failed to update Google Sheet: {e}")
        return False
    
def perform_fertilizer_analysis(df, selected_estate, nama_blok, tanggal_rencana, peilscale,
                               tanggal_terakhir, jenis_terakhir, rencana_jenis, username):
    """Performs the fertilizer analysis and returns the results."""

    try:
        tanggal_rencana = pd.to_datetime(tanggal_rencana, format="%d/%m/%Y").date()
        tanggal_terakhir = pd.to_datetime(tanggal_terakhir, format="%d/%m/%Y").date()
    except ValueError:
        tk.messagebox.showerror("Error", "Invalid date format.  Please use DD/MM/YYYY.")
        return  # Exit the function if date conversion fails

    try:
        peilscale = float(peilscale)
    except ValueError:
        tk.messagebox.showerror("Error", "Invalid Peilscale value. Please enter a number.")
        return

    # --- Your Analysis Logic Here ---
    # Example:  (Replace with your actual analysis)

    # 1.  Filter data based on estate and block (if applicable).
    estate_data = df[(df['Estate'] == selected_estate)]  # & (df['Block'] == nama_blok)]  # Add block filtering if you have a 'Block' column
    
    # 2. Get recent rainfall data.
    if not estate_data.empty:
       # Find the most recent rainfall data
        most_recent_rainfall_index = estate_data['Date'].idxmax()
        curah_hujan = estate_data.loc[most_recent_rainfall_index, 'Daily Rainfall (mm)']
    else:
        curah_hujan = 0  # Or some other default value, or handle the missing data

    # 3. Perform calculations (This is highly simplified, adapt to your needs).
    if curah_hujan > 50 and (datetime.now().date() - tanggal_terakhir).days > 7:
        status = "Allowed"
        reason = "Sufficient rainfall and time since last fertilization."
    else:
        status = "Not Allowed"
        reason = "Insufficient rainfall or too soon after last fertilization."

    recommendation = "Adjust fertilization plan based on conditions."

    return curah_hujan, status, reason, recommendation


# --- UI Functions ---

def get_date(entry_widget):
    """Creates a calendar popup and inserts the selected date into the entry widget."""
    from tkcalendar import Calendar

    if not root_exists:
        return

    def set_date():
        if not root_exists:
            return
        selected_date = cal.get_date()  #  yyyy-mm-dd
        #Convert selected_date to format dd/mm/yyyy
        try:
           date_obj = datetime.strptime(selected_date, "%Y-%m-%d")
           formatted_date = date_obj.strftime("%d/%m/%Y")
           entry_widget.delete(0, tk.END)
           entry_widget.insert(0, formatted_date)  # Insert *formatted* date
        except ValueError:
            entry_widget.delete(0, tk.END)
            entry_widget.insert(0, selected_date) #If error, insert original value

        top.destroy()

    top = tk.Toplevel(root)
    today = datetime.now(current_timezone)
    cal = Calendar(top,
                   font="Arial 10",
                   selectmode='day',
                   year=today.year,
                   month=today.month,
                   day=today.day,
                   date_pattern="yyyy-mm-dd")  # yyyy-mm-dd format
    cal.pack(pady=20)
    confirm_button = tk.Button(top, text="OK", command=set_date)
    confirm_button.pack(pady=10)
    top.transient(root)
    top.grab_set()
    top.wait_window(top)

def show_rainfall_options():
    """Displays the rainfall menu options."""
    global label_rainfall_option, back_button, current_menu, button_update_rainfall, button_add_rainfall, previous_menu

    if not root_exists:
        return

    hide_main_widgets()
    current_menu = "rainfall"
    previous_menu = "main"
    label_rainfall_option = tk.Label(root, text="Choose Rainfall Option:", font=("Arial", 12))
    label_rainfall_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    button_update_rainfall = tk.Button(root, text="Update the last Daily Rainfall (mm)", command=goto_update_rainfall,
                                        font=("Arial", 10))
    button_update_rainfall.grid(row=1, column=0, padx=10, pady=10, sticky="ew")

    button_add_rainfall = tk.Button(root, text="Add a new Daily Rainfall (mm)", command=goto_add_rainfall,
                                     font=("Arial", 10))
    button_add_rainfall.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=3, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def goto_update_rainfall():
    """Navigates to the estate selection for updating rainfall."""
    global previous_menu
    if not root_exists:
        return
    show_estate_options()
    previous_menu = "rainfall"

def goto_add_rainfall():
    """Navigates to the estate selection for adding rainfall."""
    global previous_menu, current_time_date
    if not root_exists:
        return
    show_estate_options_for_add_rainfall(current_time_date)

def submit_estate_for_analysis(selected_estate, nama_blok, tanggal_rencana, peilscale, tanggal_terakhir, jenis_terakhir, rencana_jenis):
    """Submits estate data for analysis and displays the results."""
    global previous_menu, entry_username, df
    if not root_exists:
        return

    username = entry_username.get()
    if not username:
        tk.messagebox.showerror("Error", "Please enter a username.")
        return

    # --- Call the analysis function ---
    analysis_results = perform_fertilizer_analysis(df, selected_estate, nama_blok, tanggal_rencana,
                                                  peilscale, tanggal_terakhir, jenis_terakhir,
                                                  rencana_jenis, username)

    if analysis_results:  # Proceed only if analysis was successful
        curah_hujan, status, reason, recommendation = analysis_results
        display_analysis_results(
            selected_estate, nama_blok, tanggal_rencana, peilscale, tanggal_terakhir,
            jenis_terakhir, rencana_jenis, username, curah_hujan, status, reason, recommendation
        )


def display_analysis_results(selected_estate, nama_blok, tanggal_rencana, peilscale, tanggal_terakhir,
                             jenis_terakhir, rencana_jenis, username, curah_hujan, status, reason, recommendation):
    """Displays the results of the fertilizer analysis."""
    global current_menu, label_tanggal_analisa, label_nama_user, label_curah_hujan, \
           label_status, label_reason, label_recommendation, label_selected_estate, \
           label_nama_blok, label_tanggal_rencana, label_peilscale_value, \
           label_tanggal_terakhir_value, label_jenis_terakhir_value, \
           label_rencana_jenis_value, back_to_main_button, reanalyze_button

    if not root_exists: return
    hide_estate_widgets()
    current_menu = "analysis_results"

    current_time_input = datetime.now(current_timezone)
    label_tanggal_analisa = tk.Label(root, text=f"Tanggal Analisa: {current_time_input.strftime('%Y-%m-%d %H:%M:%S')}", font=("Arial", 12))
    label_tanggal_analisa.grid(row=0, column=0, padx=10, pady=5, sticky="ew")

    label_nama_user = tk.Label(root, text=f"Nama User: {username}", font=("Arial", 12))
    label_nama_user.grid(row=1, column=0, padx=10, pady=5, sticky="ew")

    label_selected_estate = tk.Label(root, text=f"Selected Estate: {selected_estate}", font=("Arial", 12))
    label_selected_estate.grid(row=2, column=0, padx=10, pady=5, sticky="ew")

    label_nama_blok = tk.Label(root, text=f"Nama Blok: {nama_blok}", font=("Arial", 12))
    label_nama_blok.grid(row=3, column=0, padx=10, pady=5, sticky="ew")

    label_curah_hujan = tk.Label(root, text=f"Curah Hujan: {curah_hujan} mm", font=("Arial", 12))  # Add "mm"
    label_curah_hujan.grid(row=4, column=0, padx=10, pady=5, sticky="ew")

    label_peilscale_value = tk.Label(root, text=f"Nilai Peilscale: {peilscale}", font=("Arial", 12))
    label_peilscale_value.grid(row=5, column=0, padx=10, pady=5, sticky="ew")

    label_jenis_terakhir_value = tk.Label(root, text=f"Jenis Pupuk Terakhir: {jenis_terakhir}", font=("Arial", 12))
    label_jenis_terakhir_value.grid(row=6, column=0, padx=10, pady=5, sticky="ew")

    label_tanggal_terakhir_value = tk.Label(root, text=f"Tanggal Pupuk Terakhir: {tanggal_terakhir}", font=("Arial", 12))
    label_tanggal_terakhir_value.grid(row=7, column=0, padx=10, pady=5, sticky="ew")

    label_rencana_jenis_value = tk.Label(root, text=f"Rencana Jenis Pupuk: {rencana_jenis}", font=("Arial", 12))
    label_rencana_jenis_value.grid(row=8, column=0, padx=10, pady=5, sticky="ew")

    label_tanggal_rencana = tk.Label(root, text=f"Tanggal Rencana Pupuk: {tanggal_rencana}", font=("Arial", 12))
    label_tanggal_rencana.grid(row=9, column=0, padx=10, pady=5, sticky="ew")

    label_status = tk.Label(root, text=f"Status: {status}", font=("Arial", 12, "bold"))
    label_status.grid(row=10, column=0, padx=10, pady=5, sticky="ew")

    label_reason = tk.Label(root, text=f"Reason: {reason}", font=("Arial", 12))
    label_reason.grid(row=11, column=0, padx=10, pady=5, sticky="ew")

    label_recommendation = tk.Label(root, text=f"Recommendation: {recommendation}", font=("Arial", 12))
    label_recommendation.grid(row=12, column=0, padx=10, pady=5, sticky="ew")

    reanalyze_button = tk.Button(root, text="Re-analyze", command=go_to_reanalyze, font=("Arial", 10))
    reanalyze_button.grid(row=13, column=0, padx=10, pady=10)

    back_to_main_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10))
    back_to_main_button.grid(row=14, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def show_estate_options_for_analysis(fertilizer_type):
    """Displays the estate options for fertilizer analysis."""
    global label_estate_option, combobox_estate, submit_estate_button, back_button, current_menu, \
           entry_blok, entry_tanggal_rencana_pupuk, entry_peilscale, entry_tanggal_pupuk_terakhir, \
           combobox_jenis_pupuk_terakhir, combobox_rencana_jenis_pupuk, label_blok, label_tanggal_rencana_pupuk, \
           label_peilscale, label_tanggal_pupuk_terakhir, label_jenis_pupuk_terakhir, label_rencana_jenis_pupuk, \
           button_tanggal_rencana_pupuk, button_tanggal_pupuk_terakhir

    if not root_exists:
        return

    hide_main_widgets()
    current_menu = "estate_analysis"

    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=5, sticky="ew")

    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=5, sticky="ew")

    label_blok = tk.Label(root, text="Masukkan Nama Blok:", font=("Arial", 12))
    label_blok.grid(row=2, column=0, padx=10, pady=5, sticky="ew")

    entry_blok = tk.Entry(root, font=("Arial", 10))
    entry_blok.grid(row=3, column=0, padx=10, pady=5, sticky="ew")

    label_tanggal_rencana_pupuk = tk.Label(root, text="Masukkan tanggal rencana pupuk:", font=("Arial", 12))
    label_tanggal_rencana_pupuk.grid(row=4, column=0, padx=10, pady=5, sticky="ew")

    entry_tanggal_rencana_pupuk = tk.Entry(root, font=("Arial", 10))
    entry_tanggal_rencana_pupuk.grid(row=5, column=0, padx=10, pady=5, sticky="ew")

    button_tanggal_rencana_pupuk = tk.Button(root, text="Pilih Tanggal", command=lambda: get_date(entry_tanggal_rencana_pupuk))
    button_tanggal_rencana_pupuk.grid(row=5, column=1, padx=5, pady=5)

    label_tanggal_pupuk_terakhir = tk.Label(root, text="Masukkan tanggal pupuk terakhir:", font=("Arial", 12))
    label_tanggal_pupuk_terakhir.grid(row=8, column=0, padx=10, pady=5, sticky="ew")

    entry_tanggal_pupuk_terakhir = tk.Entry(root, font=("Arial", 10))
    entry_tanggal_pupuk_terakhir.grid(row=9, column=0, padx=10, pady=5, sticky="ew")

    button_tanggal_pupuk_terakhir = tk.Button(root, text="Pilih Tanggal", command=lambda: get_date(entry_tanggal_pupuk_terakhir))
    button_tanggal_pupuk_terakhir.grid(row=9, column=1, padx=5, pady=5)

    label_peilscale = tk.Label(root, text="Masukkan nilai Peilscale:", font=("Arial", 12))
    label_peilscale.grid(row=6, column=0, padx=10, pady=5, sticky="ew")

    entry_peilscale = tk.Entry(root, font=("Arial", 10))
    entry_peilscale.grid(row=7, column=0, padx=10, pady=5, sticky="ew")

    label_jenis_pupuk_terakhir = tk.Label(root, text="Masukkan jenis pupuk terakhir:", font=("Arial", 12))
    label_jenis_pupuk_terakhir.grid(row=10, column=0, padx=10, pady=5, sticky="ew")

    combobox_jenis_pupuk_terakhir = ttk.Combobox(root, values=fertilizer_type, width=30, font=("Arial", 10))
    combobox_jenis_pupuk_terakhir.grid(row=11, column=0, padx=10, pady=5, sticky="ew")

    label_rencana_jenis_pupuk = tk.Label(root, text="Masukkan rencana jenis pupuk:", font=("Arial", 12))
    label_rencana_jenis_pupuk.grid(row=12, column=0, padx=10, pady=5, sticky="ew")

    combobox_rencana_jenis_pupuk = ttk.Combobox(root, values=fertilizer_type, width=30, font=("Arial", 10))
    combobox_rencana_jenis_pupuk.grid(row=13, column=0, padx=10, pady=5, sticky="ew")

    submit_estate_button = tk.Button(root, text="Submit", command=lambda: submit_estate_for_analysis(
        combobox_estate.get(),
        entry_blok.get(),
        entry_tanggal_rencana_pupuk.get(),
        entry_peilscale.get(),
        entry_tanggal_pupuk_terakhir.get(),
        combobox_jenis_pupuk_terakhir.get(),
        combobox_rencana_jenis_pupuk.get()
    ), font=("Arial", 10))
    submit_estate_button.grid(row=14, column=0, padx=10, pady=10)

    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=15, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)
    root.columnconfigure(1, weight=0)

def go_to_reanalyze():
    """Navigates back to the estate analysis screen for re-analysis."""
    global previous_menu
    if not root_exists:
        return
    hide_analysis_results()
    show_estate_options_for_analysis(fertilizer_type)
    previous_menu = "estate_analysis"

def back_to_main():
    """Hides all widgets and recreates the main menu."""
    global previous_menu
    if not root_exists:
        return
    hide_all_widgets()
    create_main_widgets()
    previous_menu = "main"

def go_back():
    """Handles navigation back to the previous menu."""
    global previous_menu
    if not root_exists:
        return

    if previous_menu == "main":
        cancel_to_main()
    elif previous_menu == "rainfall":
        hide_estate_widgets()
        hide_rainfall_widgets()
        show_rainfall_options()
    elif previous_menu == "estate":
        hide_estate_widgets()
        hide_rainfall_data_entry_widgets()
        show_estate_options()
    elif previous_menu == "estate_analysis":
        hide_estate_widgets()
        create_main_widgets()
    elif previous_menu == "estate_add_rainfall":
        hide_estate_widgets()
        show_rainfall_options()
    elif previous_menu == "analysis_results":
        hide_analysis_results()
        show_estate_options_for_analysis(fertilizer_type)

def hide_rainfall_data_entry_widgets():
    """Hides widgets specific to the rainfall data entry screen."""
    if not root_exists:
        return

    try: label_update_rainfall.grid_forget()
    except AttributeError: pass
    try: entry_update_rainfall.grid_forget()
    except AttributeError: pass
    try: submit_update_rainfall_button.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()
    except AttributeError: pass
    try: main_menu_button.grid_forget()
    except AttributeError: pass

def hide_analysis_results():
    """Hides the widgets displaying analysis results."""
    if not root_exists: return
    try: label_tanggal_analisa.grid_forget()
    except AttributeError: pass
    try: label_nama_user.grid_forget()
    except AttributeError: pass
    try: label_curah_hujan.grid_forget()
    except AttributeError: pass
    try: label_status.grid_forget()
    except AttributeError: pass
    try: label_reason.grid_forget()
    except AttributeError: pass
    try: label_recommendation.grid_forget()
    except AttributeError: pass
    try: label_selected_estate.grid_forget()
    except AttributeError: pass
    try: label_nama_blok.grid_forget()
    except AttributeError: pass
    try: label_tanggal_rencana.grid_forget()
    except AttributeError: pass
    try: label_peilscale_value.grid_forget()
    except AttributeError: pass
    try: label_tanggal_terakhir_value.grid_forget()
    except AttributeError: pass
    try: label_jenis_terakhir_value.grid_forget()
    except AttributeError: pass
    try: label_rencana_jenis_value.grid_forget()
    except AttributeError: pass
    try: back_to_main_button.grid_forget()
    except AttributeError: pass
    try: reanalyze_button.grid_forget()
    except AttributeError: pass

    if 'current_menu' in globals():
        global current_menu
        current_menu = None

def show_estate_options_for_add_rainfall(current_time_date):
    """Displays options for adding rainfall data for a specific estate."""
    global label_estate_option, combobox_estate, submit_estate_add_rainfall_button, back_button, current_menu, entry_daily_rainfall, label_daily_rainfall, main_menu_button, previous_menu
    if not root_exists:
        return

    hide_rainfall_widgets()
    current_menu = "estate_add_rainfall"
    previous_menu = "rainfall"

    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=10)

    formatted_date = format_datetime(current_time_date)
    label_daily_rainfall = tk.Label(root, text=f"Masukkan Daily Rainfall (mm) hari ini ({formatted_date}):", font=("Arial", 12))
    label_daily_rainfall.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    entry_daily_rainfall = tk.Entry(root, font=("Arial", 10))
    entry_daily_rainfall.grid(row=3, column=0, padx=10, pady=10, sticky="ew")

    submit_estate_add_rainfall_button = tk.Button(root, text="Submit Estate",
                                                   command=lambda: submit_estate_for_add_rainfall(
                                                       combobox_estate.get(), entry_daily_rainfall.get(), formatted_date),  #Pass date
                                                   font=("Arial", 10))
    submit_estate_add_rainfall_button.grid(row=4, column=0, padx=10, pady=10)

    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=5, column=0, padx=10, pady=10)

    main_menu_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10))
    main_menu_button.grid(row=6, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def submit_estate_for_add_rainfall(selected_estate, daily_rainfall, date):
    """Submits the selected estate and rainfall data for adding to the sheet."""
    global previous_menu, df
    if not root_exists:
        return
    #Added date parameter
    if add_rainfall_data(df, selected_estate, daily_rainfall, date): #Use the function
        previous_menu = "main"
        back_to_main() # Go back to the main menu after successful update

def show_estate_options():
    """Displays the estate selection options."""
    global label_estate_option, combobox_estate, submit_estate_button, back_button, current_menu, main_menu_button, df
    if not root_exists:
        return

    hide_rainfall_widgets()
    hide_estate_widgets()
    hide_rainfall_data_entry_widgets()
    current_menu = "estate"
    label_estate_option = tk.Label(root, text="Pilih estate (Inti/Plasma):", font=("Arial", 12))
    label_estate_option.grid(row=0, column=0, padx=10, pady=10, sticky="ew")
    estate_options = ["Inti", "Plasma"]
    combobox_estate = ttk.Combobox(root, values=estate_options, width=30, font=("Arial", 10))
    combobox_estate.grid(row=1, column=0, padx=10, pady=10)
    submit_estate_button = tk.Button(root, text="Submit Estate", command=lambda: submit_estate(combobox_estate.get()),
                                     font=("Arial", 10))
    submit_estate_button.grid(row=2, column=0, padx=10, pady=10)
    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=3, column=0, padx=10, pady=10)
    main_menu_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10))
    main_menu_button.grid(row=4, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

def submit_estate(selected_estate):
    """Submits the selected estate and shows the rainfall data entry."""
    global previous_menu
    if not root_exists:
        return
    show_rainfall_data_entry(selected_estate)

def show_rainfall_data_entry(selected_estate):
    """Displays the rainfall data entry fields."""
    global previous_menu, entry_update_rainfall, label_update_rainfall, back_button, main_menu_button, submit_update_rainfall_button, entry_date

    if not root_exists:
        return

    hide_estate_widgets()
    hide_rainfall_widgets()
    hide_rainfall_data_entry_widgets()

    previous_menu = "estate"

    label_update_rainfall = tk.Label(root, text=f"Masukkan update daily rainfall (mm) untuk {selected_estate}:", font=("Arial", 12)) # Added estate name
    label_update_rainfall.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    entry_update_rainfall = tk.Entry(root, font=("Arial", 10))
    entry_update_rainfall.grid(row=3, column=0, padx=10, pady=10, sticky="ew")

    # --- Date Entry ---
    label_date = tk.Label(root, text="Masukkan tanggal (dd/mm/yyyy):", font=("Arial", 12))
    label_date.grid(row=4, column=0, padx=10, pady=5, sticky="ew")

    entry_date = tk.Entry(root, font=("Arial", 10))
    entry_date.grid(row=5, column=0, padx=10, pady=5, sticky="ew")

    button_tanggal = tk.Button(root, text="Pilih Tanggal", command=lambda: get_date(entry_date))
    button_tanggal.grid(row=5, column=1, padx=5, pady=5)

    # ------------------

    submit_update_rainfall_button = tk.Button(root, text="Submit Rainfall", command=lambda: submit_update_rainfall(selected_estate, entry_date.get()), font=("Arial", 10)) # Pass date
    submit_update_rainfall_button.grid(row=6, column=0, padx=10, pady=10)

    back_button = tk.Button(root, text="Back", command=go_back, font=("Arial", 10))
    back_button.grid(row=7, column=0, padx=10, pady=10)

    main_menu_button = tk.Button(root, text="Back to Main Menu", command=back_to_main, font=("Arial", 10))
    main_menu_button.grid(row=8, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)
    root.columnconfigure(1, weight=0) #For button date

def submit_update_rainfall(selected_estate, date):
    """Submits the updated rainfall data."""
    global previous_menu, df, entry_update_rainfall
    if not root_exists:
        return

    rainfall_value = entry_update_rainfall.get()
    # Now call update_rainfall_data, which handles the sheet update.
    if update_rainfall_data(df, selected_estate, rainfall_value, date): # Pass date and value
      back_to_main() # Go back to main menu if it's works

def hide_all_widgets():
    """Hides ALL widgets in the application."""
    if not root_exists:
        return

    for widget in root.winfo_children():
        try:
            widget.grid_forget()
        except AttributeError:
            pass

def cancel_to_main():
    """Returns to the main menu."""
    if not root_exists:
        return
    back_to_main()

def hide_main_widgets():
    """Hides the main menu widgets."""
    if not root_exists: return
    try: label_username.grid_forget()
    except AttributeError: pass
    try: entry_username.grid_forget()
    except AttributeError: pass
    try: button_input_hujan.grid_forget()
    except AttributeError: pass
    try: button_analisa_pemupukan.grid_forget()
    except AttributeError: pass
    try: exit_button.grid_forget()
    except AttributeError: pass

def hide_rainfall_widgets():
    """Hides widgets related to rainfall options."""
    if not root_exists: return
    try: label_rainfall_option.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()
    except AttributeError: pass
    try: submit_estate_add_rainfall_button.grid_forget()
    except AttributeError: pass
    try: button_update_rainfall.grid_forget()
    except AttributeError: pass
    try: button_add_rainfall.grid_forget()
    except AttributeError: pass
    try: label_update_rainfall.grid_forget()
    except AttributeError: pass
    try: entry_update_rainfall.grid_forget()
    except AttributeError: pass
    try: submit_update_rainfall_button.grid_forget()
    except AttributeError: pass
    try: main_menu_button.grid_forget()
    except AttributeError: pass
    try: label_date.grid_forget() #hide label date
    except AttributeError: pass
    try: entry_date.grid_forget() #Hide entry date
    except AttributeError: pass
    try: button_tanggal.grid_forget() #Hide button tanggal
    except AttributeError: pass

    global previous_menu
    previous_menu = "rainfall"

    if 'current_menu' in globals() :
        global current_menu
        current_menu = None

def hide_estate_widgets():
    """Hides widgets related to estate selection and analysis."""
    global current_menu

    if not root_exists: return

    try: label_estate_option.grid_forget()
    except AttributeError: pass
    try: combobox_estate.grid_forget()
    except AttributeError: pass
    try: submit_estate_button.grid_forget()
    except AttributeError: pass
    try: back_button.grid_forget()
    except AttributeError: pass
    try: main_menu_button.grid_forget()
    except AttributeError: pass
    try: label_blok.grid_forget()
    except AttributeError: pass
    try: entry_blok.grid_forget()
    except AttributeError: pass
    try: label_tanggal_rencana_pupuk.grid_forget()
    except AttributeError: pass
    try: entry_tanggal_rencana_pupuk.grid_forget()
    except AttributeError: pass

    if 'current_menu' in globals() and current_menu == "estate_analysis":
        try: button_tanggal_rencana_pupuk.grid_forget()
        except AttributeError: pass
        try: label_peilscale.grid_forget()
        except AttributeError: pass
        try: entry_peilscale.grid_forget()
        except AttributeError: pass
        try: label_tanggal_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: entry_tanggal_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: button_tanggal_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: label_jenis_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: combobox_jenis_pupuk_terakhir.grid_forget()
        except AttributeError: pass
        try: label_rencana_jenis_pupuk.grid_forget()
        except AttributeError: pass
        try: combobox_rencana_jenis_pupuk.grid_forget()
        except AttributeError: pass

    try: submit_estate_add_rainfall_button.grid_forget()
    except AttributeError: pass
    try: entry_daily_rainfall.grid_forget()
    except AttributeError: pass
    try: label_daily_rainfall.grid_forget()
    except AttributeError: pass
    try: label_date.grid_forget() #hide label date
    except AttributeError: pass
    try: entry_date.grid_forget() #Hide entry date
    except AttributeError: pass
    try: button_tanggal.grid_forget() #Hide button tanggal
    except AttributeError: pass

    if 'current_menu' in globals():
        current_menu = None

def create_main_widgets():
    """Creates the main menu widgets."""
    global label_username, entry_username, submit_button, previous_menu, current_menu, back_button, exit_button, button_input_hujan, button_analisa_pemupukan
    if not root_exists:
        return
    root.geometry("500x400")

    current_menu = "main"
    label_username = tk.Label(root, text="Enter Username:", font=("Arial", 12))
    label_username.grid(row=0, column=0, padx=10, pady=10, sticky="ew")

    entry_username = tk.Entry(root, font=("Arial", 10))
    entry_username.grid(row=1, column=0, padx=10, pady=10, sticky="ew")

    button_input_hujan = tk.Button(
        root, text="1. Input Data Hujan", command=goto_input_hujan, font=("Arial", 12)
    )
    button_input_hujan.grid(row=2, column=0, padx=10, pady=10, sticky="ew")

    button_analisa_pemupukan = tk.Button(
        root,
        text="2. Analisa Pemupukan",
        command=goto_analisa_pemupukan,
        font=("Arial", 12),
    )
    button_analisa_pemupukan.grid(row=3, column=0, padx=10, pady=10, sticky="ew")

    exit_button = tk.Button(root, text="Exit", command=on_closing, font=("Arial", 10))
    exit_button.grid(row=5, column=0, padx=10, pady=10)

    root.columnconfigure(0, weight=1)

    previous_menu = None
    back_button = None

def goto_input_hujan():
    """Navigates to the rainfall input menu."""
    global previous_menu
    if not root_exists: return
    previous_menu = "main"
    hide_all_widgets()
    show_rainfall_options()

def goto_analisa_pemupukan():
    """Navigates to the fertilizer analysis menu."""
    global previous_menu, fertilizer_type
    if not root_exists: return
    previous_menu = "main"
    hide_all_widgets()
    show_estate_options_for_analysis(fertilizer_type)

def disable_buttons():
    """Disables all interactive buttons."""
    global back_button, submit_rainfall_button, submit_estate_button, exit_button, submit_estate_add_rainfall_button, button_input_hujan, button_analisa_pemupukan, button_update_rainfall, button_add_rainfall, reanalyze_button, main_menu_button, submit_update_rainfall_button, button_tanggal_rencana_pupuk, button_tanggal_pupuk_terakhir, button_tanggal

    if not root_exists:
        return

    try:
        if back_button: back_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_rainfall_button: submit_rainfall_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_estate_button: submit_estate_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if exit_button: exit_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_estate_add_rainfall_button: submit_estate_add_rainfall_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_input_hujan: button_input_hujan.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_analisa_pemupukan: button_analisa_pemupukan.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_update_rainfall: button_update_rainfall.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_add_rainfall: button_add_rainfall.config(state="disabled")
    except tk.TclError: pass
    try:
        if reanalyze_button: reanalyze_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if main_menu_button: main_menu_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if submit_update_rainfall_button: submit_update_rainfall_button.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_tanggal_rencana_pupuk: button_tanggal_rencana_pupuk.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_tanggal_pupuk_terakhir: button_tanggal_pupuk_terakhir.config(state="disabled")
    except tk.TclError: pass
    try:
        if button_tanggal: button_tanggal.config(state="disabled")
    except tk.TclError: pass

def on_closing():
    """Handles the window closing event."""
    global root_exists
    root_exists = False
    disable_buttons()
    root.destroy()

def main_process():
    """Main function to set up and run the application."""
    global root, previous_menu, root_exists, current_menu, \
           submit_button, back_button, submit_rainfall_button, \
           submit_estate_button, exit_button, \
           label_estate_option, combobox_estate, entry_blok, \
           label_tanggal_rencana_pupuk, entry_tanggal_rencana_pupuk, \
           label_peilscale, entry_peilscale, label_tanggal_pupuk_terakhir, \
           entry_tanggal_pupuk_terakhir, label_jenis_pupuk_terakhir, \
           combobox_jenis_pupuk_terakhir, label_rencana_jenis_pupuk, \
           combobox_rencana_jenis_pupuk, label_rainfall_option, \
           combobox_rainfall, submit_estate_add_rainfall_button, \
           entry_daily_rainfall, label_username, entry_username, \
           label_menu_choice, label_daily_rainfall, label_blok, button_input_hujan, button_analisa_pemupukan, button_update_rainfall, button_add_rainfall, label_tanggal_analisa, label_nama_user, label_curah_hujan, label_status, label_reason, label_recommendation, label_selected_estate, label_nama_blok, label_tanggal_rencana, label_peilscale_value, label_tanggal_terakhir_value, label_jenis_terakhir_value, label_rencana_jenis_value, back_to_main_button, reanalyze_button, current_time_date, main_menu_button, label_update_rainfall, entry_update_rainfall, submit_update_rainfall_button, df, entry_date, label_date, button_tanggal

    root = tk.Tk()
    root.title("Fertilizer Analysis")
    root.state('zoomed')
    previous_menu = None
    root_exists = True
    current_menu = None
    df = pd.DataFrame() # Initialize df here

    # Initialize all widget variables to None
    label_username = None
    entry_username = None
    exit_button = None
    label_rainfall_option = None
    combobox_rainfall = None
    submit_rainfall_button = None  # No longer directly used
    back_button = None
    label_estate_option = None
    combobox_estate = None
    submit_estate_button = None
    entry_blok = None
    label_tanggal_rencana_pupuk = None
    entry_tanggal_rencana_pupuk = None
    label_peilscale = None
    entry_peilscale = None
    label_tanggal_pupuk_terakhir = None
    entry_tanggal_pupuk_terakhir = None
    label_jenis_pupuk_terakhir = None
    combobox_jenis_pupuk_terakhir = None
    label_rencana_jenis_pupuk = None
    combobox_rencana_jenis_pupuk = None
    submit_estate_add_rainfall_button = None
    entry_daily_rainfall = None
    label_daily_rainfall = None
    label_blok = None
    button_input_hujan = None
    button_analisa_pemupukan = None
    button_update_rainfall = None
    button_add_rainfall = None
    label_tanggal_analisa = None
    label_nama_user = None
    label_curah_hujan = None
    label_status = None
    label_reason = None
    label_recommendation = None
    label_selected_estate = None
    label_nama_blok = None
    label_tanggal_rencana = None
    label_peilscale_value = None
    label_tanggal_terakhir_value = None
    label_jenis_terakhir_value = None
    label_rencana_jenis_value = None
    back_to_main_button = None
    reanalyze_button = None
    main_menu_button = None
    label_update_rainfall = None
    entry_update_rainfall = None
    submit_update_rainfall_button = None
    label_date = None  # Initialize label_date
    entry_date = None   # Initialize entry_date
    button_tanggal = None # Initialize button_tanggal

    root.protocol("WM_DELETE_WINDOW", on_closing)
    root.columnconfigure(0, weight=1)

    # --- Load Data ---
    df = load_data(sheet_url, json_path) # Load data at the start
    if df.empty: #Check if the df is empty
        tk.messagebox.showerror("Error", "Failed to load data from the spreadsheet.  Please check your connection and credentials.")
        root.destroy() #Close the app if can't load the data
        return # Exit the function

    create_main_widgets()
    root.mainloop()

if __name__ == "__main__":
    fertilizer_type = ["NPK 13", "NPK 15", "NPK 12", "Dolomite", "Urea", "MOP", "HGFB", "CuSO4", "Zincop Chelated", "Kieserite", "RP", "Kaptan", "TSP"]

    current_timezone = pytz.timezone('Asia/Jakarta')
    date_input = datetime.now(current_timezone)
    current_time_date = datetime.now(current_timezone).date()

    main_process()